# പാഠം 17 - Foundry Local ഉം Qwen ഉം ഉപയോഗിച്ച് ലോക്കൽ AI ഏജന്റുകൾ സൃഷ്‌ടിക്കൽ

ഈ നോട്ട്‌ബുക്കിൽ നിങ്ങൾ **പൂർണമായി നിങ്ങളുടെ ജോലി തൊഴിലാളിയിൽ പ്രവർത്തിക്കുന്ന ലോക്കൽ എഞ്ചിനീയറിങ്ങ് അസിസ്റ്റന്റ്** നിർമ്മിക്കും. ഏത് നിമിഷത്തിലും ക്ലൗഡ് ഇൻഫറൻസ് ഉപയോഗിക്കാറില്ല. അസിസ്റ്റന്റ് ചെയ്യുന്നത്:

1. Foundry Local വഴി Qwen ഫംഗ്ഷൻ കോളിങ് മുഖേന **ടൂളുകൾ വിളിക്കുക**.
2. സാൻഡ്‌ബോക്സ് ചെയ്ത പ്രോജക്റ്റ് ഡയറക്ടറിയിലുൾപ്പെടെയുള്ള **ഫയലുകൾ പട്ടികപ്പെടുത്തുകയും വായിക്കുകയും** ചെയ്യുക.
3. സംഗീതമുള്ള മെട്രിക്‌സ് ഉപയോഗിച്ച് **കോഡ് വിശകലനം** ചെയ്യുക.
4. ലോക്കൽ RAG (Chroma) ഉപയോഗിച്ച് **ഡോക്യുമെന്റേഷൻ തിരയുക**.
5. **ലോക്കൽ MCP സേർവർ** ഉപയോഗിക്കുക (അന്വേഷിച്ചു കാണാതെ യാതൊരു ക്രമീകരണവും ഇല്ലെങ്കിൽ സുഖത്തോടെ എടുക്കും).

ഏജന്റ് കോഡ് ക്ലൗഡ് പാഠ്യപുസ്തകങ്ങൾക്കു വളരെ നൂറു സമാനമാണ് — വെറും ക്ലയന്റ് എൻഡ്‌പോയിന്റ് ക്ലൗഡിൽനിന്ന് `localhost`ലേക്ക് മാറുന്നു.


## സജ്ജീകരണം

ഈ നോട്ട്ബുക്ക് റൺ ചെയ്യുന്നതിനുമുന്‍പ്:

1. **Microsoft Foundry Local ഇൻസ്റ്റാൾ ചെയ്യുക** (നിങ്ങളുടെ ഓപ്പറേറ്റിംഗ് സിസ്റ്റത്തിനായി [ഡോക്യുമെന്റേഷൻ](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) കാണുക).
2. **ഒരു Qwen മോഡൽ ഡൗൺലോഡ് ചെയ്യുകയും ആരംഭിക്കുകയും ചെയ്യുക:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. താഴെ കൊടുക്കുന്ന Python പാക്കേജുകൾ ഇൻസ്റ്റാൾ ചെയ്യുക.

എല്ലാം ലോക്കലിലേ തന്നെ പ്രവർത്തിക്കുന്നു. ഏകദേശം ~8 ജിബി റാമുള്ള ഒരു യന്ത്രം യാഥാർത്ഥ്യപരമായ ഏറ്റവും താഴ്ന്നതായുണ്ടാകണം.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. ഫൗണ്ട്രി ലോക്കലുമായി ബന്ധിപ്പിക്കുക

`FoundryLocalManager` മോഡൽ ആവശ്യമുണ്ടെങ്കിൽ ഡൗൺലോഡ് ചെയ്യുന്നു, ലോക്കൽ സർവീസ് ആരംഭിക്കുന്നു, പിന്നെ നമ്മുക്ക് **OpenAI-ഉപയോഗയോഗ്യമായ എന്റ്പോയിന്റ്** നൽകുന്നു. അതിനു ശേഷം നാം സ്റ്റാൻഡേർഡ് OpenAI SDK-യെ അതിലേക്കു apunt ചെയ്യുന്നു. API കീ ഒരു ലോക്കൽ പ്ലേസ്ഹോൾഡറാണ് — ക്ലൗഡ് ക്രെഡൻഷ്യൽ ഉൾപ്പെടുത്തിയിട്ടില്ല.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. ലോക്കൽ ടൂളുകൾ (സാൻഡ്‌ബോക്‌സ് ചെയ്ത ഫയൽ ഓപ്പറേഷനുകൾ)

ഞങ്ങൾ ഡിസ്കിൽ ഒരു ചെറിയ സൊമ്പിൾ പ്രോജക്ട് സൃഷ്ടിച്ച്, പിന്നീട് ആ പ്രോജക്ട് റൂട്ടിൽ സാങ്കേതികമായി ബന്ധപ്പെട്ട ടൂളുകൾ നിർവചിക്കുന്നു. സാൻഡ്‌ബോക്‌സ് പരിശോധന ലോക്കലായി പോലും കാര്യമാണ്: ഒരു ടൂൾ ഏത് പാതകളും വായിക്കുമ്പോൾ നിങ്ങളുടെ ഉപയോക്താവിന്റെ അനുമതികളോടെ പ്രവർത്തിക്കും, അതിനാൽ നിങ്ങൾ ആ പാതകളിൽ എന്തും സ്പർശിക്കാനാകും.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. ക്രോമയുമായി ലോക്കൽ RAG

ഞങ്ങൾ ഡോക്യുമെന്റേഷൻ സ്നിപ്പെറ്റുകളുടെ ചെറിയ ഒരു സെറ്റ് ലോക്കൽ ക്രോമ<|vq_lbr_audio_86563|><|vq_lbr_audio_75999|><|vq_lbr_audio_30546|><|vq_lbr_audio_80996|><|vq_lbr_audio_38829|><|vq_lbr_audio_38829|><|vq_lbr_audio_38829|><|vq_lbr_audio_49659|><|vq_lbr_audio_65485|><|vq_lbr_audio_65485|><|vq_lbr_audio_65485|><|vq_lbr_audio_95714|><|vq_lbr_audio_65485|><|vq_lbr_audio_65485|><|vq_lbr_audio_43894|><|vq_lbr_audio_55451|><|vq_lbr_audio_46647|><|vq_lbr_audio_95962|><|vq_lbr_audio_96342|><|vq_lbr_audio_99852|><|vq_lbr_audio_81596|><|vq_lbr_audio_125642|><|vq_lbr_audio_13496|><|vq_lbr_audio_114765|><|vq_lbr_audio_75268|><|vq_lbr_audio_128292|><|vq_lbr_audio_78440|><|vq_lbr_audio_51120|><|vq_lbr_audio_29645|><|vq_lbr_audio_87652|><|vq_lbr_audio_92926|><|vq_lbr_audio_100193|><|vq_lbr_audio_105367|><|vq_lbr_audio_81020|><|vq_lbr_audio_43888|><|vq_lbr_audio_61404|><|vq_lbr_audio_98943|><|vq_lbr_audio_129544|><|vq_lbr_audio_95523|><|vq_lbr_audio_7067|><|vq_lbr_audio_39277|><|vq_lbr_audio_47213|><|vq_lbr_audio_103682|><|vq_lbr_audio_95828|><|vq_lbr_audio_123530|><|vq_lbr_audio_99510|><|vq_lbr_audio_41813|><|vq_lbr_audio_128785|><|vq_lbr_audio_106407|><|vq_lbr_audio_104144|><|vq_lbr_audio_53044|><|vq_lbr_audio_14348|><|vq_lbr_audio_85321|><|vq_lbr_audio_31954|><|vq_lbr_audio_15410|><|vq_lbr_audio_109350|><|vq_lbr_audio_80190|><|vq_lbr_audio_15729|><|vq_lbr_audio_111626|><|vq_lbr_audio_56034|><|vq_lbr_audio_30881|><|vq_lbr_audio_43072|><|vq_lbr_audio_94601|><|vq_lbr_audio_31628|><|vq_lbr_audio_114714|><|vq_lbr_audio_92887|><|vq_lbr_audio_111941|><|vq_lbr_audio_33444|><|vq_lbr_audio_63258|><|vq_lbr_audio_118938|><|vq_lbr_audio_82168|><|vq_lbr_audio_67175|><|vq_lbr_audio_26745|><|vq_lbr_audio_65918|><|vq_lbr_audio_35856|><|vq_lbr_audio_59508|><|vq_lbr_audio_107790|><|vq_lbr_audio_17070|><|vq_lbr_audio_125429|><|vq_lbr_audio_60514|><|vq_lbr_audio_91886|><|vq_lbr_audio_80045|><|vq_lbr_audio_70607|><|vq_lbr_audio_117894|><|vq_lbr_audio_46591|><|vq_lbr_audio_9851|><|vq_lbr_audio_6478|><|vq_lbr_audio_49479|><|vq_lbr_audio_66452|><|vq_lbr_audio_34221|><|vq_lbr_audio_28763|><|vq_lbr_audio_30902|><|vq_lbr_audio_42221|><|vq_lbr_audio_125549|><|vq_lbr_audio_46785|><|vq_lbr_audio_46785|><|vq_lbr_audio_32264|><|vq_lbr_audio_29093|><|vq_lbr_audio_38399|><|vq_lbr_audio_91996|><|vq_lbr_audio_37894|><|vq_lbr_audio_11685|><|vq_lbr_audio_113222|><|vq_lbr_audio_109838|><|vq_lbr_audio_19863|><|vq_lbr_audio_94983|><|vq_lbr_audio_46449|><|vq_lbr_audio_52062|><|vq_lbr_audio_45215|><|vq_lbr_audio_108686|><|vq_lbr_audio_69337|><|vq_lbr_audio_61811|><|vq_lbr_audio_66881|><|vq_lbr_audio_77286|><|vq_lbr_audio_101273|><|vq_lbr_audio_26788|><|vq_lbr_audio_117081|><|vq_lbr_audio_110333|><|vq_lbr_audio_73142|><|vq_lbr_audio_59743|><|vq_lbr_audio_103727|><|vq_lbr_audio_21571|><|vq_lbr_audio_1185|><|vq_lbr_audio_45037|><|vq_lbr_audio_125415|><|vq_lbr_audio_42944|><|vq_lbr_audio_130899|><|vq_lbr_audio_101047|><|vq_lbr_audio_46860|><|vq_lbr_audio_87927|><|vq_lbr_audio_72758|><|vq_lbr_audio_62731|><|vq_lbr_audio_113540|><|vq_lbr_audio_27130|><|vq_lbr_audio_104503|><|vq_lbr_audio_91907|><|vq_lbr_audio_3113|><|vq_lbr_audio_73423|><|vq_lbr_audio_31000|><|vq_lbr_audio_23050|><|vq_lbr_audio_104485|><|vq_lbr_audio_6744|><|vq_lbr_audio_50683|><|vq_lbr_audio_9500|><|vq_lbr_audio_101118|><|vq_lbr_audio_22056|><|vq_lbr_audio_123810|><|vq_lbr_audio_64155|><|vq_lbr_audio_71186|><|vq_lbr_audio_52063|><|vq_lbr_audio_83999|><|vq_lbr_audio_24366|><|vq_lbr_audio_98156|><|vq_lbr_audio_63825|><|vq_lbr_audio_79146|><|vq_lbr_audio_4556|><|vq_lbr_audio_65249|><|vq_lbr_audio_22883|><|vq_lbr_audio_6282|><|vq_lbr_audio_61444|><|vq_lbr_audio_107930|><|vq_lbr_audio_81642|><|vq_lbr_audio_116823|><|vq_lbr_audio_91528|><|vq_lbr_audio_29477|><|vq_lbr_audio_5427|><|vq_lbr_audio_70119|><|vq_lbr_audio_57489|><|vq_lbr_audio_93331|><|vq_lbr_audio_94098|><|vq_lbr_audio_83379|><|vq_lbr_audio_38752|><|vq_lbr_audio_89198|><|vq_lbr_audio_113997|><|vq_lbr_audio_121243|><|vq_lbr_audio_44211|><|vq_lbr_audio_90888|><|vq_lbr_audio_124793|><|vq_lbr_audio_45170|><|vq_lbr_audio_13881|><|vq_lbr_audio_54828|><|vq_lbr_audio_23043|><|vq_lbr_audio_88249|><|vq_lbr_audio_47040|><|vq_lbr_audio_22770|><|vq_lbr_audio_87170|><|vq_lbr_audio_23130|><|vq_lbr_audio_44580|><|vq_lbr_audio_22882|><|vq_lbr_audio_35666|><|vq_lbr_audio_96726|><|vq_lbr_audio_36093|><|vq_lbr_audio_99736|><|vq_lbr_audio_64024|><|vq_lbr_audio_65756|><|vq_lbr_audio_112804|><|vq_lbr_audio_24054|><|vq_lbr_audio_106245|><|vq_lbr_audio_111404|><|vq_lbr_audio_120188|><|vq_lbr_audio_60248|><|vq_lbr_audio_95146|><|vq_lbr_audio_22110|><|vq_lbr_audio_24252|><|vq_lbr_audio_12706|><|vq_lbr_audio_115534|><|vq_lbr_audio_126599|><|vq_lbr_audio_26247|><|vq_lbr_audio_115264|><|vq_lbr_audio_74521|><|vq_lbr_audio_16107|><|vq_lbr_audio_123324|><|vq_lbr_audio_5114|><|vq_lbr_audio_60561|><|vq_lbr_audio_61553|><|vq_lbr_audio_2152|><|vq_lbr_audio_27639|><|vq_lbr_audio_37413|><|vq_lbr_audio_104609|><|vq_lbr_audio_12465|><|vq_lbr_audio_25590|><|vq_lbr_audio_5315|><|vq_lbr_audio_45102|><|vq_lbr_audio_54907|><|vq_lbr_audio_61685|><|vq_lbr_audio_96468|><|vq_lbr_audio_26745|><|vq_lbr_audio_19863|><|vq_lbr_audio_77938|><|vq_lbr_audio_77938|><|vq_lbr_audio_22213|><|vq_lbr_audio_23081|><|vq_lbr_audio_96109|><|vq_lbr_audio_10483|><|vq_lbr_audio_95698|><|vq_lbr_audio_113009|><|vq_lbr_audio_67806|><|vq_lbr_audio_44171|><|vq_lbr_audio_69795|><|vq_lbr_audio_86681|><|vq_lbr_audio_2289|><|vq_lbr_audio_124793|><|vq_lbr_audio_41891|><|vq_lbr_audio_38829|><|vq_lbr_audio_49659|><|vq_lbr_audio_65485|><|vq_lbr_audio_12631|><|vq_lbr_audio_9727|><|vq_lbr_audio_83423|><|vq_lbr_audio_67307|><|vq_lbr_audio_47017|><|vq_lbr_audio_21351|><|vq_lbr_audio_69198|><|vq_lbr_audio_4241|><|vq_lbr_audio_117050|><|vq_lbr_audio_13286|><|vq_lbr_audio_130777|><|vq_lbr_audio_99901|><|vq_lbr_audio_3778|><|vq_lbr_audio_24772|><|vq_lbr_audio_94574|><|vq_lbr_audio_83980|><|vq_lbr_audio_81625|><|vq_lbr_audio_31008|><|vq_lbr_audio_47618|><|vq_lbr_audio_45324|><|vq_lbr_audio_55721|><|vq_lbr_audio_123566|><|vq_lbr_audio_87797|><|vq_lbr_audio_75459|><|vq_lbr_audio_106984|><|vq_lbr_audio_100304|><|vq_lbr_audio_121176|><|vq_lbr_audio_126878|><|vq_lbr_audio_83400|><|vq_lbr_audio_18488|><|vq_lbr_audio_7732|><|vq_lbr_audio_21067|><|vq_lbr_audio_13437|><|vq_lbr_audio_5454|><|vq_lbr_audio_42373|><|vq_lbr_audio_8386|><|vq_lbr_audio_38031|><|vq_lbr_audio_70666|><|vq_lbr_audio_91287|><|vq_lbr_audio_41019|><|vq_lbr_audio_128812|><|vq_lbr_audio_121487|><|vq_lbr_audio_89275|><|vq_lbr_audio_46479|><|vq_lbr_audio_54907|><|vq_lbr_audio_77938|><|vq_lbr_audio_128441|><|vq_lbr_audio_59505|><|vq_lbr_audio_23465|><|vq_lbr_audio_62107|><|vq_lbr_audio_124499|><|vq_lbr_audio_15649|><|vq_lbr_audio_79066|><|vq_lbr_audio_23477|><|vq_lbr_audio_116682|><|vq_lbr_audio_55135|><|vq_lbr_audio_12217|><|vq_lbr_audio_4424|><|vq_lbr_audio_105643|><|vq_lbr_audio_118774|><|vq_lbr_audio_95714|><|vq_lbr_audio_115799|><|vq_lbr_audio_79924|><|vq_lbr_audio_125165|><|vq_lbr_audio_34515|><|vq_lbr_audio_15511|><|vq_lbr_audio_37681|><|vq_lbr_audio_69108|><|vq_lbr_audio_50633|><|vq_lbr_audio_2699|><|vq_lbr_audio_110963|><|vq_lbr_audio_49635|><|vq_lbr_audio_73512|><|vq_lbr_audio_18716|><|vq_lbr_audio_125784|><|vq_lbr_audio_103491|><|vq_lbr_audio_111456|><|vq_lbr_audio_9278|><|vq_lbr_audio_22542|><|vq_lbr_audio_42755|><|vq_lbr_audio_106648|><|vq_lbr_audio_9390|><|vq_lbr_audio_111165|><|vq_lbr_audio_111248|><|vq_lbr_audio_88123|><|vq_lbr_audio_126145|><|vq_lbr_audio_55691|><|vq_lbr_audio_31050|><|vq_lbr_audio_67306|><|vq_lbr_audio_31395|><|vq_lbr_audio_83019|><|vq_lbr_audio_122433|><|vq_lbr_audio_8800|><|vq_lbr_audio_128006|><|vq_lbr_audio_65485|><|vq_lbr_audio_66777|><|vq_lbr_audio_107958|><|vq_lbr_audio_67256|><|vq_lbr_audio_103249|><|vq_lbr_audio_111471|><|vq_lbr_audio_126201|><|vq_lbr_audio_86938|><|vq_lbr_audio_3490|><|vq_lbr_audio_89417|><|vq_lbr_audio_7280|><|vq_lbr_audio_84031|><|vq_lbr_audio_117819|><|vq_lbr_audio_87696|><|vq_lbr_audio_100991|><|vq_lbr_audio_87135|><|vq_lbr_audio_83661|><|vq_lbr_audio_81038|><|vq_lbr_audio_15577|><|vq_lbr_audio_94144|><|vq_lbr_audio_90729|><|vq_lbr_audio_32659|><|vq_lbr_audio_125847|><|vq_lbr_audio_79680|><|vq_lbr_audio_34880|><|vq_lbr_audio_112053|><|vq_lbr_audio_41753|><|vq_lbr_audio_71535|><|vq_lbr_audio_66833|><|vq_lbr_audio_83968|><|vq_lbr_audio_24264|><|vq_lbr_audio_57878|><|vq_lbr_audio_121899|><|vq_lbr_audio_27634|><|vq_lbr_audio_119304|><|vq_lbr_audio_115564|><|vq_lbr_audio_122543|><|vq_lbr_audio_124803|><|vq_lbr_audio_8210|><|vq_lbr_audio_22699|><|vq_lbr_audio_17370|><|vq_lbr_audio_18317|><|vq_lbr_audio_42650|><|vq_lbr_audio_126602|><|vq_lbr_audio_57926|><|vq_lbr_audio_53274|><|vq_lbr_audio_58347|><|vq_lbr_audio_63282|><|vq_lbr_audio_130799|><|vq_lbr_audio_66976|><|vq_lbr_audio_84298|><|vq_lbr_audio_108006|><|vq_lbr_audio_3252|><|vq_lbr_audio_125247|><|vq_lbr_audio_36838|><|vq_lbr_audio_10887|><|vq_lbr_audio_49659|><|vq_lbr_audio_18822|><|vq_lbr_audio_49659|><|vq_lbr_audio_58759|><|vq_lbr_audio_38829|><|vq_lbr_audio_40374|><|vq_lbr_audio_40374|><|vq_lbr_audio_120837|><|vq_lbr_audio_124555|><|vq_lbr_audio_11882|><|vq_lbr_audio_81436|><|vq_lbr_audio_295|><|vq_lbr_audio_18822|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_54989|><|vq_lbr_audio_54989|><|vq_lbr_audio_124555|><|vq_lbr_audio_104070|><|vq_lbr_audio_94732|><|vq_lbr_audio_110479|><|vq_lbr_audio_111843|><|vq_lbr_audio_59602|><|vq_lbr_audio_121994|><|vq_lbr_audio_24631|><|vq_lbr_audio_85777|><|vq_lbr_audio_119205|><|vq_lbr_audio_5152|><|vq_lbr_audio_32338|><|vq_lbr_audio_30608|><|vq_lbr_audio_125210|><|vq_lbr_audio_8879|><|vq_lbr_audio_49659|><|vq_lbr_audio_78299|><|vq_lbr_audio_11744|><|vq_lbr_audio_13535|><|vq_lbr_audio_18822|><|vq_lbr_audio_18822|><|vq_lbr_audio_18822|><|vq_lbr_audio_33344|><|vq_lbr_audio_33715|><|vq_lbr_audio_17405|><|vq_lbr_audio_104569|><|vq_lbr_audio_85996|><|vq_lbr_audio_56469|><|vq_lbr_audio_119079|><|vq_lbr_audio_35955|><|vq_lbr_audio_23636|><|vq_lbr_audio_59848|><|vq_lbr_audio_93225|><|vq_lbr_audio_4556|><|vq_lbr_audio_70770|><|vq_lbr_audio_27419|><|vq_lbr_audio_36449|><|vq_lbr_audio_59655|><|vq_lbr_audio_69244|><|vq_lbr_audio_937|><|vq_lbr_audio_65485|><|vq_lbr_audio_54907|><|vq_lbr_audio_1031|><|vq_lbr_audio_99491|><|vq_lbr_audio_21310|><|vq_lbr_audio_96123|><|vq_lbr_audio_54596|><|vq_lbr_audio_104919|><|vq_lbr_audio_17397|><|vq_lbr_audio_31520|><|vq_lbr_audio_45796|><|vq_lbr_audio_44940|><|vq_lbr_audio_21980|><|vq_lbr_audio_25548|><|vq_lbr_audio_46631|><|vq_lbr_audio_68184|><|vq_lbr_audio_13395|><|vq_lbr_audio_53131|><|vq_lbr_audio_117246|><|vq_lbr_audio_22157|><|vq_lbr_audio_10980|><|vq_lbr_audio_1355|><|vq_lbr_audio_110698|><|vq_lbr_audio_129107|><|vq_lbr_audio_65858|><|vq_lbr_audio_13969|><|vq_lbr_audio_106449|><|vq_lbr_audio_69974|><|vq_lbr_audio_57067|><|vq_lbr_audio_127517|><|vq_lbr_audio_101870|><|vq_lbr_audio_77938|><|vq_lbr_audio_81116|><|vq_lbr_audio_60453|><|vq_lbr_audio_125665|><|vq_lbr_audio_48543|><|vq_lbr_audio_94384|><|vq_lbr_audio_82563|><|vq_lbr_audio_51612|><|vq_lbr_audio_58310|><|vq_lbr_audio_33702|><|vq_lbr_audio_54907|><|vq_lbr_audio_24689|><|vq_lbr_audio_7403|><|vq_lbr_audio_20848|><|vq_lbr_audio_56906|><|vq_lbr_audio_17022|><|vq_lbr_audio_94063|><|vq_lbr_audio_8649|><|vq_lbr_audio_39889|><|vq_lbr_audio_58426|><|vq_lbr_audio_102152|><|vq_lbr_audio_23485|><|vq_lbr_audio_67363|><|vq_lbr_audio_95714|><|vq_lbr_audio_65485|><|vq_lbr_audio_68669|><|vq_lbr_audio_118791|><|vq_lbr_audio_95714|><|vq_lbr_audio_1355|><|vq_lbr_audio_27614|><|vq_lbr_audio_118457|><|vq_lbr_audio_113428|><|vq_lbr_audio_83745|><|vq_lbr_audio_38606|><|vq_lbr_audio_115713|><|vq_lbr_audio_27812|><|vq_lbr_audio_87279|><|vq_lbr_audio_44312|><|vq_lbr_audio_85840|><|vq_lbr_audio_55764|><|vq_lbr_audio_101153|><|vq_lbr_audio_37066|><|vq_lbr_audio_27587|><|vq_lbr_audio_12745|><|vq_lbr_audio_84519|><|vq_lbr_audio_6952|><|vq_lbr_audio_15833|><|vq_lbr_audio_72999|><|vq_lbr_audio_14891|><|vq_lbr_audio_56101|><|vq_lbr_audio_116428|><|vq_lbr_audio_13778|><|vq_lbr_audio_7451|><|vq_lbr_audio_40905|><|vq_lbr_audio_73427|><|vq_lbr_audio_85368|><|vq_lbr_audio_106843|><|vq_lbr_audio_117558|><|vq_lbr_audio_24366|><|vq_lbr_audio_3996|><|vq_lbr_audio_97900|><|vq_lbr_audio_11175|><|vq_lbr_audio_105752|><|vq_lbr_audio_12049|><|vq_lbr_audio_66294|><|vq_lbr_audio_61390|><|vq_lbr_audio_95714|><|vq_lbr_audio_72272|><|vq_lbr_audio_82260|><|vq_lbr_audio_92183|><|vq_lbr_audio_75350|><|vq_lbr_audio_42087|><|vq_lbr_audio_84602|><|vq_lbr_audio_119726|><|vq_lbr_audio_124075|><|vq_lbr_audio_107283|><|vq_lbr_audio_16101|><|vq_lbr_audio_7588|><|vq_lbr_audio_52707|><|vq_lbr_audio_90194|><|vq_lbr_audio_116049|><|vq_lbr_audio_54989|><|vq_lbr_audio_53504|><|vq_lbr_audio_13778|><|vq_lbr_audio_124555|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_60696|><|vq_lbr_audio_46308|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_44267|><|vq_lbr_audio_2375|><|vq_lbr_audio_13666|><|vq_lbr_audio_16512|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113100|><|vq_lbr_audio_2375|><|vq_lbr_audio_21257|><|vq_lbr_audio_62737|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_31370|><|vq_lbr_audio_2375|><|vq_lbr_audio_81445|><|vq_lbr_audio_2375|><|vq_lbr_audio_64954|><|vq_lbr_audio_16471|><|vq_lbr_audio_84130|><|vq_lbr_audio_24366|><|vq_lbr_audio_46308|><|vq_lbr_audio_21007|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_101153|><|vq_lbr_audio_104969|><|vq_lbr_audio_14822|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_2362|><|vq_lbr_audio_13666|><|vq_lbr_audio_118755|><|vq_lbr_audio_2375|><|vq_lbr_audio_52768|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_129544|><|vq_lbr_audio_20896|><|vq_lbr_audio_124555|><|vq_lbr_audio_77938|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_105391|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_105391|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_105391|><|vq_lbr_audio_105391|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_105391|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_58759|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_58759|><|vq_lbr_audio_124555|><|vq_lbr_audio_105391|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_53504|><|vq_lbr_audio_124555|><|vq_lbr_audio_114096|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_124555|><|vq_lbr_audio_13778|><|vq_lbr_audio_61829|><|vq_lbr_audio_42102|><|vq_lbr_audio_24366|><|vq_lbr_audio_4676|><|vq_lbr_audio_60696|><|vq_lbr_audio_90995|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_60696|><|vq_lbr_audio_27236|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_90995|><|vq_lbr_audio_16258|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_60696|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_77401|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_82522|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_29895|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_59738|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_113801|><|vq_lbr_audio_120258|><|vq_lbr_audio_24366|><|vq_lbr_audio_2062|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_39774|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_14348|><|vq_lbr_audio_123408|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_113801|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_129730|><|vq_lbr_audio_71983|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_31441|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_60744|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_71983|><|vq_lbr_audio_71983|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_65999|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_82917|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_50373|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|><|vq_lbr_audio_123808|><|vq_lbr_audio_24366|><|vq_lbr_audio_24366|>


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. ടൂൾ-കോളിംഗ് ലൂപ്പ്  

ഇപ്പോൾ നാം OpenAI ടൂളുകളുടെ സ്കീമ ഉപയോഗിച്ച് മോഡലിനൊപ്പം ടൂളുകൾ രജിസ്റ്റർ ചെയ്ത് സ്റ്റാൻഡേർഡ് ടൂൾ-കോളിംഗ് ലൂപ്പ് നടത്തും — മോഡൽ ഒരു ടൂൾ ആവശ്യപ്പെടുന്നു, നാം അത് ലൊക്കലായി പ്രവർത്തിപ്പിക്കുകയും ഫലം തിരികെ നൽക്കുകയും ചെയ്യുന്നു, മോഡൽ അന്തിമ ഉത്തരം നൽകുന്നത് വരെ ഇത് പുനരാവൃതമാക്കുന്നു. ക്വെന്റെ വിശ്വസനീയമായ ഫംഗ്ഷൻ കോളിംഗ് ഇത് ഡിവൈസിൽ പ്രവർത്തിപ്പിക്കാൻ സഹായിക്കുന്നു.  


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. ലൊക്കൽ MCP (ഐച്ഛികം)

MCP ഒരു ക്ലൗഡ് സർവീസ് അല്ല, ട്രാൻസ്പോർട്ട് ആണ് — ഒരു MCP സർവർ `stdio`-യെ മുൾക്കൊണ്ടു ഒരു ലൊക്കൽ പ്രോസസ്സായി പ്രവർത്തിക്കാം. താഴെയുള്ള സെല്ലിൽ നിങ്ങൾക്ക് നിങ്ങളെക്കുറിച്ചുള്ള ഒരു MCP സർവറുമായി (ഉദാഹരണത്തിന് ഒരു ഫയൽസിസ്റ്റം സർവർ) എങ്ങനെ കണക്ട് ചെയ്യാമെന്ന് കാണിക്കുന്നു. `LOCAL_MCP_COMMAND` സജ്ജമാക്കിയിട്ടില്ലെങ്കിൽ ഇത് സൌമ്യമായി ഒഴിവാക്കുന്നു, അതിനാൽ നോട്ട് ബുക്ക് പൂർത്തിയായി പ്രവർത്തിക്കുന്നു.

സുരക്ഷാ കുറിപ്പ്: ഒരു ലൊക്കൽ MCP സർവർ നിങ്ങളുടെ യൂസർ അനുമതികളോടെ പ്രവർത്തിക്കുന്നു. അത് ഒരു പ്രോജക്ട് ഡയറക്ടറിയിലേക്കാണ് പരിമിതപ്പെടുത്തുക, അതിന്റെ ഔട്ട്‌പുട്ടുകൾ പ്രവർത്തിക്കുന്നതിന് മുമ്പ് പരിശോദിക്കുക.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## സാരാംശം

നിങ്ങൾ പൂർണ്ണമായും നിങ്ങളുടെ യന്ത്രത്തിൽ പ്രവർത്തിക്കുന്ന ഒരു എഞ്ചിനീയറിംഗ് അസിസ്റ്റന്റ് നിർമ്മിച്ചു:

- **Foundry Local** ഒരു OpenAI-സ്‌തരാനുകൂല എൻഡ്‌പോയിന്റിനടുത്ത് **Qwen** മോഡൽ വിതരണം ചെയ്തു — അതിനാൽ ഏജന്റ് കോഡ് ക്ലൗഡ് പാഠങ്ങളോട് പൊരുത്തപ്പെടുന്നു.
- **Sandboxed tools** പ്രോജക്ട് ഡയറക്ടറി വിടാതെ ഏജന്റിന് ഫയൽ ആക്‌സസ് കൂടി കോഡ് വിശകലനം നൽകി.
- **Chroma** ഡോക്യുമെന്റേഷനിൽ **ലോകൽ RAG** നൽകി.
- **Local MCP** MCP പദ്ധതിസംരംഭം ഓഫ്‌ലൈൻ വീണ്ടും ഉപയോഗിക്കാൻ കാണിച്ചു.

ഏതു സമയത്തും ക്ലൗഡ് ഇൻഫറൻസ് ഉപയോഗിച്ചില്ല.

### സവിചാരം

ലോക്കൽ ഏജന്റിനെ വിപുലീകരിക്കുക:

1. **മുല്ടിപ്പിൾ MCP സർവറുകളുമായി ജോലി ചെയ്യുക** — ഫയൽസിസ്റ്റം സർവർ, ഗിറ്റ് സർവർ എന്നിവ ബന്ധിപ്പിച്ച് ഏജന്റ് അവയിൽ നിന്നും തിരഞ്ഞെടുക്കാൻ അനുവദിക്കുക.
2. **ലോകൽ മെമ്മറി ഉപയോഗിക്കുക** — അസിസ്റ്റന്റ് നോട്ട്ബുക്ക് റീസ്റ്റാർട്ടുകൾക്കിടയിൽ മുമ്പത്തെ സംവാദം ഓർമ്മിക്കാവുന്ന വിധം ചെറു സംഭാഷണ ചരിത്രം ഡിസ്കിൽ സേവ് ചെയ്യുക.
3. **ലോകൽ മൾട്ടി-ഏജന്റ് ഓർക്കസ്ട്രേഷൻ പിന്തുണയ്‌ക്കുക** — രണ്ടാം ലോക്കൽ ഏജന്റ് (ഉദാ. ഒരു റിവ്യൂവർ) ചേർത്ത് ഇരുവരും ഒരു ടാസ്‌കിൽ ചേർന്ന് പ്രവർത്തിക്കാൻ അനുവദിക്കുക.

അടുത്ത പാഠത്തിൽ നിങ്ങൾ വിതരണത്തിലെ AI ഏജന്റുകൾ എങ്ങനെ സുരക്ഷിതമാക്കാമെന്ന് പഠിക്കും.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
